# Phase 7 - Predictive scenario simulation

**Important framing.** This notebook answers *what does the model predict if this input changes?* It does **not** prove *this input caused the demand change*. The simulator is mechanical: it perturbs feature columns and asks the trained quantile model again.

Pipeline:

1. Load the Phase 4 full-horizon feature table and the Phase 6 quantile-model bundle.
2. Pick an origin date (defaults to the latest backtest origin we have a model for).
3. Run six default scenarios (price +/-10%, event_on, snap_on, momentum +/-20%) plus any custom ones.
4. Persist long forecasts and per-scenario summary.
5. Plot a base-vs-scenario fan for one example id.

All real logic lives in `seercast.scenario.{adjustments, simulator, comparison}` and `seercast.visualization.plots`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from seercast.training.run_scenarios import run as run_scenarios
from seercast.visualization.plots import plot_scenario_fan, plot_multiple_scenario_fans

REPO_ROOT

## 1. Run the default scenario set

In [ ]:
result = run_scenarios()
forecasts = result['forecasts']
summary = result['summary']
origin_date = result['origin_date']
print(f'origin: {origin_date.date()}')
summary

## 2. Inspect deltas

Long format: one row per (scenario, id, horizon).

In [ ]:
forecasts.head()

## 3. Base vs scenario fan for one example id

Pick the highest-volume id at this origin. The fill bands are p10-p90 intervals; the lines are p50 medians.

In [ ]:
top_id = (
    forecasts[forecasts['scenario'] == forecasts['scenario'].iloc[0]]
    .groupby('id')['base_p50'].sum().idxmax()
)
print(f'eyeballing id: {top_id}')
scen_to_show = 'price_-10pct' if 'price_-10pct' in forecasts['scenario'].unique() else forecasts['scenario'].iloc[0]
ax = plot_scenario_fan(forecasts, id_=top_id, scenario=scen_to_show)
plt.tight_layout()
plt.show()

## 4. All scenarios for one id

In [ ]:
fig = plot_multiple_scenario_fans(forecasts, id_=top_id)
plt.show()

## 5. Custom scenarios

Build any combination by pairing adjustment functions with `functools.partial`, then chaining if needed.

In [ ]:
from functools import partial
from seercast.scenario import (
    apply_event_scenario, apply_price_scenario, chain,
)

custom = {
    'event_with_price_drop': chain(
        partial(apply_event_scenario, event_on=True),
        partial(apply_price_scenario, price_change_pct=-15.0),
    ),
}
custom_result = run_scenarios(scenarios=custom)
custom_result['summary']

**Reminder one more time.** Scenario forecasts describe what the trained model would output under altered inputs. They are not causal estimates and out-of-distribution scenarios (e.g. very large price moves) should be treated as extrapolation.

**Project implementation roadmap is paused after this phase.**